In [1]:
!pip install torch_geometric torch_scatter torch_sparse torch_cluster torch_spline_conv
!pip install rdkit

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.7/63.7 kB 5.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 108.0/108.0 kB 8.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 210.0/210.0 kB 15.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.5/54.5 kB 4.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 59.2 MB/s eta 0:00:00
  Created wheel for torch_scatter: filename=torch_scatter-2.1.2-cp312-cp312-linux_x86_64.whl size=664632 sha256=af4cf54c9fede5114b9e8ef0714794d2d1ec3627a87b2f8fb3d947fda8f00f17
  Stored in directory: /root/.cache/pip/wheels/84/20/50/44800723f57cd798630e77b3ec83bc80bd26a1e3dc3a672ef5
  Created wheel for torch_sparse: filename=torch_sparse-0.6.18-cp312-cp312-linux_x86_64.whl size=1222874 sha256=33993e42cab31a6074a658df0685a07746feeb1aa

In [2]:
!pip install python-docx

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 14.5 MB/s eta 0:00:00


In [3]:
!pip install pubchempy

In [ ]:
import docx
import pubchempy as pcp
import pandas as pd
import time

## Compound Name Extraction

In [5]:
def extract_compounds_from_docx(file_path):
    doc = docx.Document(file_path)
    # Table 1 is usually the second table (index 1) in your document
    table = doc.tables[1]

    names = []
    for row in table.rows[1:]: # Skip the header row
        # Column 2 (index 2) contains the compound names
        name = row.cells[2].text.strip()
        if name:
            names.append(name)
    return list(set(names)) # Remove duplicates

compound_list = extract_compounds_from_docx('OLA 2025 Publication UPD1.docx')
print(f"Extracted {len(compound_list)} unique compound names.")

PackageNotFoundError: Package not found at 'OLA 2025 Publication UPD1.docx'

In [ ]:
def get_final_smiles(file_path):
    doc = docx.Document(file_path)
    # Extract names from the table in the new file
    names = []
    table = doc.tables[0]
    for row in table.rows[1:]: # Skip header
        name = row.cells[2].text.strip()
        if name: names.append(name)

    print(f"Found {len(names)} major bioactives. Starting lookup...")

    results = []
    for name in names:
        try:
            time.sleep(0.5) # Prevent API lockout
            compounds = pcp.get_compounds(name, 'name')
            if compounds:
                results.append({"Name": name, "SMILES": compounds[0].canonical_smiles})
                print(f" Found: {name}")
            else:
                results.append({"Name": name, "SMILES": "MANUAL_CHECK"})
                print(f" Missing: {name}")
        except:
            results.append({"Name": name, "SMILES": "ERROR"})

    return pd.DataFrame(results)

# Run the lookup
phytochemicals_df = get_final_smiles('Ola nano table 2.docx')
phytochemicals_df.to_csv('phytochemical_smiles.csv', index=False)

In [ ]:


# Define the exact mapping for the 13 major bioactives
chemical_master_list = {
    "2, 6-Octadienal, 3, 7-dimethyl-(Z)": "CC(=CCCC(=CC=O)C)C",
    "Citral": "CC(=CCCC(=CC=O)C)C",
    "Cis-3-Hexenyl cis-3-hexenoate": "CCC=CCC O C(=O)CC=CCC", # Simplified canonical form
    "Trans-2, 7-Dimethyl-4, 6-octadien-2-ol": "CC(C)(O)C=CC=C(C)C",
    "3,8,11-Trioxatetracyclo[4.4.1.0(2,4).0(7,9)]undecane (1α,2α,4α,6α,7β,9β)": "C1C2C(O1)C3C(O3)C(O2)C(C)C",
    "3-Hexene, 2-methyl-(Z)-": "CCC=C(C)C",
    "Hexadecanoic acid, methyl ester": "CCCCCCCCCCCCCCCC(=O)OC",
    "n-Hexadecanoic acid": "CCCCCCCCCCCCCCCC(=O)O",
    "cis-Vaccenic acid": "CCCCCCC=CCCCCCCCCC(=O)O",
    "6, 11-Dimethyl-2, 6, 10-dedecatrien-1-ol": "CC(C)=CCCC(C)=CCCC(C)CO",
    "Heptadecanoic acid, 16-methyl-, methyl ester": "CC(C)CCCCCCCCCCCCCC(=O)OC",
    "9, 17-Octadecadienal, (Z)-": "CC=CCCCCCCCC=CCCCCCCCC=O",
    "Oleic Acid": "CCCCCCCCC=CCCCCCCCC(=O)O"
}

def finalize_phytochemical_data(file_path):
    print("Step 1: Extracting names from document...")
    doc = docx.Document(file_path)
    extracted_names = []
    # Targeted extraction from the GC-MS table
    table = doc.tables[0]
    for row in table.rows[1:]:
        name = row.cells[2].text.strip()
        if name:
            extracted_names.append(name)

    print(f"Step 2: Building dictionary for {len(extracted_names)} compounds...")
    final_data = []
    for name in extracted_names:
        smiles = chemical_master_list.get(name, "NOT_FOUND")
        final_data.append({"Name": name, "SMILES": smiles})

    # Create and save the dataframe
    df = pd.DataFrame(final_data)
    df.to_csv('phytochemical_dictionary_final.csv', index=False)

    print("\n--- Process Complete ---")
    print(df)
    print("\nFile saved as: phytochemical_dictionary_final.csv")
    return df
try:
    final_df = finalize_phytochemical_data('Ola nano table 2.docx')
except Exception as e:
    print(f"Error: {e}. Please ensure the .docx file is uploaded to the Colab sidebar.")

In [12]:
from rdkit import Chem
from rdkit.Chem import Draw
import os
from PIL import Image as PILImage # Explicitly import Image from PIL with an alias
import matplotlib.pyplot as plt # Import matplotlib for potential figure saving
from IPython import display # Import for checking IPython.display.Image
import base64 # For decoding base64 data if present

# Define the exact mapping for the 13 major bioactives
chemical_master_list = {
    "2, 6-Octadienal, 3, 7-dimethyl-(Z)": "CC(=CCCC(=CC=O)C)C",
    "Citral": "CC(=CCCC(=CC=O)C)C",
    "Cis-3-Hexenyl cis-3-hexenoate": "CCC=CCC O C(=O)CC=CCC", # Simplified canonical form
    "Trans-2, 7-Dimethyl-4, 6-octadien-2-ol": "CC(C)(O)C=CC=C(C)C",
    "3,8,11-Trioxatetracyclo[4.4.1.0(2,4).0(7,9)]undecane (1α,2α,4α,6α,7β,9β)": "C1C2C(O1)C3C(O3)C(O2)C(C)C",
    "3-Hexene, 2-methyl-(Z)-": "CCC=C(C)C",
    "Hexadecanoic acid, methyl ester": "CCCCCCCCCCCCCCCC(=O)OC",
    "n-Hexadecanoic acid": "CCCCCCCCCCCCCCCC(=O)O",
    "cis-Vaccenic acid": "CCCCCCC=CCCCCCCCCC(=O)O",
    "6, 11-Dimethyl-2, 6, 10-dedecatrien-1-ol": "CC(C)=CCCC(C)=CCCC(C)CO",
    "Heptadecanoic acid, 16-methyl-, methyl ester": "CC(C)CCCCCCCCCCCCCC(=O)OC",
    "9, 17-Octadecadienal, (Z)-": "CC=CCCCCCCCC=CCCCCCCCC=O",
    "Oleic Acid": "CCCCCCCCC=CCCCCCCCC(=O)O"
}

# Create the directory if it doesn't exist
os.makedirs('exports/figures/', exist_ok=True)

# Example for Citral (one of your key drivers)
citral_mol = Chem.MolFromSmiles("CC(=CCCC(=CCO)C)C") # Use the actual SMILES from your list
Draw.MolToFile(citral_mol, 'exports/figures/citral_2d_structure.png', size=(300, 300))

# To save all 13 at once in a grid
list_of_your_13_smiles = list(chemical_master_list.values())
your_list_of_names = list(chemical_master_list.keys())

mols = [Chem.MolFromSmiles(s) for s in list_of_your_13_smiles]
grid = Draw.MolsToGridImage(mols, molsPerRow=4, subImgSize=(200, 200), legends=your_list_of_names)

# Determine the type of the grid object and save accordingly
if isinstance(grid, plt.Figure):
    grid.savefig('exports/figures/phytochemical_grid.png')
elif isinstance(grid, PILImage.Image):
    grid.save('exports/figures/phytochemical_grid.png')
elif isinstance(grid, display.Image) and hasattr(grid, 'data'):
    # If it's an IPython.display.Image, it contains base64 encoded PNG data
    try:
        img_data = base64.b64decode(grid.data)
        with open('exports/figures/phytochemical_grid.png', 'wb') as f:
            f.write(img_data)
    except Exception as e:
        print(f"Could not decode and save IPython.display.Image: {e}")
else:
    print(f"Warning: Unknown image object type {type(grid)}. Could not save grid image.")

print(" 2D molecular structures saved!")

 2D molecular structures saved!
